# Store and product-family demand forecasting

This notebook forecasts sales separately for every store × product-family series in Kaggle Store Sales. It builds only past-demand lag/rolling features, uses promotion and holiday inputs only when known for the target dates, compares seasonal-naive with LightGBM and XGBoost when available, and evaluates identical rolling 16-day windows. The final two windows are held out from model selection.

In Kaggle settings choose GPU and two T4 accelerators if offered. The tree models train on CPU in this notebook; the extra T4 GPUs are not required for this workflow. The expected accuracy gains come from separate panel histories, causal lag features and covariates. Add Input: the Store Sales competition data. Outputs and model files are zipped at the end; click the link to download locally.

In [ ]:
from pathlib import Path
import json, time, zipfile, warnings, platform, pickle, gc
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, FileLink, Markdown
warnings.filterwarnings('ignore'); SEED=42; HORIZON=16
INPUT=Path('/kaggle/input/competitions/store-sales-time-series-forecasting') if Path('/kaggle/input/competitions/store-sales-time-series-forecasting').exists() else Path('/kaggle/input')
OUT=Path('/kaggle/working/demand_forecast_output') if Path('/kaggle/working').exists() else Path('./demand_forecast_output'); OUT.mkdir(parents=True,exist_ok=True)
TRAIN_PATH=next(iter(sorted(INPUT.rglob('train.csv'))),None)
if TRAIN_PATH is None: raise FileNotFoundError('train.csv not found under '+str(INPUT))
TEST_PATH=next(iter(sorted(INPUT.rglob('test.csv'))),None); HOLIDAY_PATH=next(iter(sorted(INPUT.rglob('holidays_events.csv'))),None); STORES_PATH=next(iter(sorted(INPUT.rglob('stores.csv'))),None)
print('Train:',TRAIN_PATH,'Test:',TEST_PATH,'Holidays:',HOLIDAY_PATH,'Stores:',STORES_PATH)
train_raw=pd.read_csv(TRAIN_PATH); print('Train shape:',train_raw.shape); display(train_raw.head())
required={'date','store_nbr','family','sales'}
if not required.issubset(train_raw.columns): raise ValueError('Expected columns '+str(required)+'; found '+str(train_raw.columns.tolist()))
if TEST_PATH is not None: TEST_RAW=pd.read_csv(TEST_PATH)
if HOLIDAY_PATH is not None: HOLIDAYS_RAW=pd.read_csv(HOLIDAY_PATH)
if STORES_PATH is not None: STORES_RAW=pd.read_csv(STORES_PATH)
if TEST_PATH is not None: TEST_RAW['date']=pd.to_datetime(TEST_RAW.date)
if 'onpromotion' not in train_raw: train_raw['onpromotion']=0
if TEST_PATH is not None and 'onpromotion' not in TEST_RAW: TEST_RAW['onpromotion']=0

In [ ]:
# Inspect source completeness. Missing dates remain missing (not silently converted into zero sales).
train_raw['date']=pd.to_datetime(train_raw.date,errors='coerce'); train_raw['sales']=pd.to_numeric(train_raw.sales,errors='coerce')
train_raw=train_raw.dropna(subset=['date','sales']).drop_duplicates(subset=['date','store_nbr','family'],keep='last')
train_raw['onpromotion']=pd.to_numeric(train_raw.onpromotion,errors='coerce')
dates=pd.date_range(train_raw.date.min(),train_raw.date.max(),freq='D'); observed=sorted(train_raw.date.unique()); missing_dates=dates.difference(pd.DatetimeIndex(observed))
print('Coverage:',dates.min().date(),'to',dates.max().date(),'| missing calendar dates:',[str(x.date()) for x in missing_dates])
print('Negative demand rows:',int((train_raw.sales<0).sum()),'Duplicate date/store/family keys after cleanup:',int(train_raw.duplicated(['date','store_nbr','family']).sum()))
print('Entity counts: stores=',train_raw.store_nbr.nunique(),'families=',train_raw.family.nunique(),'series=',train_raw[['store_nbr','family']].drop_duplicates().shape[0])
display(train_raw.groupby('family').sales.agg(['mean','std','min','max']).sort_values('mean',ascending=False).head(10))
fig,ax=plt.subplots(figsize=(13,4)); train_raw.groupby('date').sales.sum().plot(ax=ax); ax.set(title='Total sales across entities (context only)',ylabel='Sales'); plt.tight_layout(); plt.savefig(OUT/'aggregate_context.png'); plt.show()
# Mark public holidays by date. Source's transferred events are retained as calendar events; document interpretation.
holiday_dates=set()
if HOLIDAY_PATH is not None:
 holidays=pd.read_csv(HOLIDAY_PATH); holidays['date']=pd.to_datetime(holidays.date,errors='coerce'); holiday_dates=set(holidays.loc[holidays.date.notna(),'date'].dt.normalize())
print('Holiday source rows:',0 if HOLIDAY_PATH is None else len(holidays),'unique holiday dates:',len(holiday_dates))
# Create a full date grid per known store-family. Missing target observations remain NaN.
entities=train_raw[['store_nbr','family']].drop_duplicates().sort_values(['store_nbr','family']).reset_index(drop=True)
grid=entities.merge(pd.DataFrame({'date':dates}),how='cross')
actual=train_raw[['date','store_nbr','family','sales','onpromotion']]
panel=grid.merge(actual,on=['date','store_nbr','family'],how='left',validate='one_to_one')
panel['is_holiday']=panel.date.dt.normalize().isin(holiday_dates).astype('int8')
if STORES_PATH is not None and 'store_nbr' in STORES_RAW:
 store_meta=STORES_RAW[['store_nbr']+[c for c in ['state','type','cluster'] if c in STORES_RAW.columns]].drop_duplicates('store_nbr')
 panel=panel.merge(store_meta,on='store_nbr',how='left',validate='many_to_one')
else:
 panel['state']='unknown'; panel['type']='unknown'; panel['cluster']=-1
panel['family_code']=panel.family.astype('category').cat.codes.astype('int16')
panel['state_code']=panel.state.astype('category').cat.codes.astype('int16')
panel['type_code']=panel.type.astype('category').cat.codes.astype('int8')
print('Panel rows:',len(panel),'unobserved sales cells preserved as NaN:',int(panel.sales.isna().sum()),'zero-sales records:',int((panel.sales==0).sum()))
panel=panel.sort_values(['store_nbr','family','date']).reset_index(drop=True)
panel.loc[panel.sales.notna() & (panel.date >= panel.date.max()-pd.Timedelta(days=365)), ['date','store_nbr','family','sales']].to_csv(OUT/'historical_sales_by_store_family.csv',index=False)
entity_meta=panel.drop_duplicates(['store_nbr','family']).set_index(['store_nbr','family'])[['family_code','state_code','type_code','cluster']].to_dict('index')
# Keep one year of actuals in the Kaggle bundle for the dashboard's recent history.


In [ ]:
# Causal panel features: every target lag/rolling statistic is from earlier dates in the same series.
KEYS=['store_nbr','family']; grp=panel.groupby(KEYS,sort=False,observed=True)
for lag in [1,7,14,28]: panel[f'lag_{lag}']=grp.sales.shift(lag)
for w in [7,28]: panel[f'roll_mean_{w}']=grp.sales.transform(lambda x:x.shift(1).rolling(w,min_periods=w).mean())
panel['dow']=panel.date.dt.dayofweek.astype('int8'); panel['day']=panel.date.dt.day.astype('int8'); panel['month']=panel.date.dt.month.astype('int8'); panel['weekofyear']=panel.date.dt.isocalendar().week.astype('int16'); panel['year']=panel.date.dt.year.astype('int16'); panel['weekend']=(panel.dow>=5).astype('int8')
FEATURES=['store_nbr','family_code','state_code','type_code','cluster','lag_1','lag_7','lag_14','lag_28','roll_mean_7','roll_mean_28','dow','day','month','weekofyear','year','weekend','is_holiday','onpromotion']
features=panel[['date','family','sales']+FEATURES].copy()
# No imputation of target. Rows with absent targets or insufficient lag history are excluded from model fitting.
trainable=features.loc[features.sales.notna() & features[FEATURES].notna().all(axis=1)].copy()
print('Training examples:',len(trainable),'from',trainable.date.min().date(),'to',trainable.date.max().date())
display(trainable[FEATURES+['sales']].head())
# Create 5 sequential 16-day evaluation windows. First 3 are validation; last 2 are final test.
unique_dates=pd.DatetimeIndex(sorted(train_raw.date.unique())); nfolds=5; start_idx=len(unique_dates)-nfolds*HORIZON
if start_idx<100: raise ValueError('Not enough dates for five rolling windows.')
folds=[]
for i in range(nfolds):
 origin=unique_dates[start_idx+i*HORIZON-1]; window=unique_dates[start_idx+i*HORIZON:start_idx+(i+1)*HORIZON]
 folds.append({'fold':i+1,'origin':origin,'dates':window,'split':'validation' if i<3 else 'test'})
print(pd.DataFrame([{'fold':f['fold'],'split':f['split'],'origin':str(f['origin'].date()),'from':str(f['dates'][0].date()),'through':str(f['dates'][-1].date())} for f in folds]).to_string(index=False))

In [ ]:
# Metrics include MAE/RMSE/sMAPE and competition-style RMSLE; baseline is scored on every same window.
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_squared_log_error
def score(y,p):
 y=np.asarray(y,dtype=float); p=np.maximum(0,np.asarray(p,dtype=float)); den=np.abs(y)+np.abs(p)
 return {'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p))),'sMAPE_pct':float(np.mean(np.where(den==0,0,2*np.abs(y-p)/den))*100),'RMSLE':float(np.sqrt(mean_squared_log_error(np.maximum(0,y),p)))}
def seasonal_predictions(hist,ids,target_dates):
 lookup={(int(r.store_nbr),r.family,r.date):r.sales for r in hist.itertuples()}; out=[]
 for dt in target_dates:
  prev=dt-pd.Timedelta(days=7)
  for store,family in ids:
   val=lookup.get((int(store),family,prev),np.nan)
   if pd.isna(val):
    candidates=[(d,v) for (s,f,d),v in lookup.items() if s==int(store) and f==family and d<dt and pd.notna(v)]
    val=candidates[-1][1] if candidates else 0.0
   val=max(0,float(val)); out.append((dt,int(store),family,val)); lookup[(int(store),family,dt)]=val
 return pd.DataFrame(out,columns=['date','store_nbr','family','prediction'])
def future_covariates(target_dates):
 # Train rows provide observed promotions for backtest dates; competition test.csv provides known future promotion plans.
 cols=['date','store_nbr','family','onpromotion']
 base=train_raw[cols].copy()
 if TEST_PATH is not None:
  future=TEST_RAW[cols].copy(); base=pd.concat([base,future],ignore_index=True)
 return base.drop_duplicates(['date','store_nbr','family'],keep='last')
covariates=future_covariates(None)
def recursive_forecast(model, history, target_dates, ids):
 # Predict one date at a time and append predictions to history for subsequent recursive lags.
 h={(int(store),family):pd.Series(g.sales.values,index=pd.DatetimeIndex(g.date)) for (store,family),g in history.groupby(KEYS,sort=False)}
 promos=covariates.loc[covariates.date.isin(target_dates)]
 promo_lookup={(int(r.store_nbr),r.family,r.date):r.onpromotion for r in promos.itertuples()}
 outputs=[]
 for dt in target_dates:
  batch=[]
  for store,family in ids:
   ts=h[(int(store),family)]
   at=lambda k: ts.get(dt-pd.Timedelta(days=k),np.nan)
   recent=ts.loc[(ts.index<dt)&(ts.index>=dt-pd.Timedelta(days=7))]
   recent28=ts.loc[(ts.index<dt)&(ts.index>=dt-pd.Timedelta(days=28))]
   promo=promo_lookup.get((int(store),family,dt),np.nan)
   if pd.isna(promo): promo=0.0
   m=entity_meta[(int(store),family)]
   batch.append({'date':dt,'store_nbr':int(store),'family':family,'family_code':m['family_code'],'state_code':m['state_code'],'type_code':m['type_code'],'cluster':m['cluster'],
                 'lag_1':at(1),'lag_7':at(7),'lag_14':at(14),'lag_28':at(28),'roll_mean_7':recent.mean(),'roll_mean_28':recent28.mean(),
                 'dow':dt.dayofweek,'day':dt.day,'month':dt.month,'weekofyear':int(dt.isocalendar().week),'year':dt.year,
                 'weekend':int(dt.dayofweek>=5),'is_holiday':int(dt.normalize() in holiday_dates),'onpromotion':float(promo)})
  day=pd.DataFrame(batch)
  day['forecast']=np.maximum(0,model.predict(day[FEATURES]))
  outputs.append(day)
  for row in day.itertuples():
   h[(int(row.store_nbr),row.family)].loc[row.date]=row.forecast
 return pd.concat(outputs,ignore_index=True),h
print('Validation folds use historical promotion values; competition future promotions are joined from test.csv when available.')

In [ ]:
# Model factories. Training is pooled across series; identifiers let one model share patterns across stores/families.
models={}
try:
 import lightgbm as lgb
 models['LightGBM']=lambda:lgb.LGBMRegressor(n_estimators=500,learning_rate=.05,num_leaves=63,max_depth=-1,subsample=.8,colsample_bytree=.85,reg_lambda=2,random_state=SEED,n_jobs=-1,verbosity=-1)
 print('LightGBM',lgb.__version__)
except Exception as e: print('LightGBM unavailable:',repr(e))
try:
 import xgboost as xgb
 models['XGBoost']=lambda:xgb.XGBRegressor(n_estimators=500,max_depth=8,learning_rate=.05,subsample=.8,colsample_bytree=.85,reg_lambda=2,objective='reg:squarederror',tree_method='hist',enable_categorical=False,random_state=SEED,n_jobs=-1)
 print('XGBoost',xgb.__version__)
except Exception as e: print('XGBoost unavailable:',repr(e))
if not models: raise ImportError('Install LightGBM or XGBoost in the Kaggle session, then rerun.')
print('Candidates:',list(models))
# Limit fit examples to reduce Kaggle RAM/time; random sample is from permitted past training dates only.
MAX_TRAIN_ROWS=1500000
def fit_candidate(name, origin):
 fit=trainable.loc[trainable.date<=origin]
 if len(fit)>MAX_TRAIN_ROWS: fit=fit.sample(MAX_TRAIN_ROWS,random_state=SEED)
 model=models[name](); model.fit(fit[FEATURES],fit.sales); return model
ids=list(map(tuple,entities[['store_nbr','family']].itertuples(index=False,name=None)))
print('Rows per fit capped at',MAX_TRAIN_ROWS,'; no row after a fold origin is included in that fold training.')

In [ ]:
# Run identical rolling windows for seasonal-naive and each available ML model.
records=[]; fitted_for_final={}; runtime=[]; entity_records=[]
def save_entity_scores(frame, fold, split, model_name):
 for level,col in [('family','family'),('store','store_nbr')]:
  for key,g in frame.groupby(col):
   entity_records.append({'fold':fold,'split':split,'model':model_name,'level':level,'entity':str(key),**score(g.actual,g.forecast)})
for fold in folds:
 origin=fold['origin']; target_dates=fold['dates']; hist=panel.loc[panel.date<=origin].copy()
 actual=panel.loc[panel.date.isin(target_dates),['date','store_nbr','family','sales']].dropna(subset=['sales']).rename(columns={'sales':'actual'})
 basepred=seasonal_predictions(hist,target_dates,ids).rename(columns={'prediction':'forecast'})
 baseline=actual.merge(basepred,on=['date','store_nbr','family'],how='inner')
 bm=score(baseline.actual,baseline.forecast)
 records.append({'fold':fold['fold'],'split':fold['split'],'model':'SeasonalNaive_7',**bm}); save_entity_scores(baseline,fold['fold'],fold['split'],'SeasonalNaive_7')
 print('Fold',fold['fold'],fold['split'],'baseline',bm)
 for name in models:
  t0=time.time(); model=fit_candidate(name,origin); predrows,histories=recursive_forecast(model,hist,target_dates,ids);
  merged=actual.merge(predrows[['date','store_nbr','family','forecast']],on=['date','store_nbr','family'],how='inner'); sc=score(merged.actual,merged.forecast)
  records.append({'fold':fold['fold'],'split':fold['split'],'model':name,**sc}); save_entity_scores(merged,fold['fold'],fold['split'],name); runtime.append({'fold':fold['fold'],'model':name,'seconds':time.time()-t0})
  print('Fold',fold['fold'],fold['split'],name,sc)
  if fold['fold']==folds[-1]['fold']: fitted_for_final[name]=model
  del model,predrows,merged; gc.collect()
scores=pd.DataFrame(records); display(scores)
summary=scores.groupby(['split','model'])[['MAE','RMSE','sMAPE_pct','RMSLE']].agg(['mean','std']).round(4); display(summary)
(OUT/'rolling_window_scores.csv').write_text(scores.to_csv(index=False))
(OUT/'rolling_window_summary.csv').write_text(summary.to_csv())
pd.DataFrame(runtime).to_csv(OUT/'training_times.csv',index=False)
pd.DataFrame(entity_records).to_csv(OUT/'store_family_metrics_by_fold.csv',index=False)
# Select only from validation windows. Final two windows remain the held-out test score.
valmean=scores.loc[scores.split=='validation'].groupby('model').MAE.mean().sort_values(); selected=valmean.index[0]
print('Selected using validation-window mean MAE only:',selected,'| validation ranking:',valmean.to_dict())
test_summary=scores.loc[scores.split=='test'].groupby('model')[['MAE','RMSE','sMAPE_pct','RMSLE']].mean().sort_values('MAE'); print('Held-out test window mean metrics (not used for selection):'); display(test_summary)

In [ ]:
# Final 16-day forecast from latest history. Competition test.csv supplies known promotion plans when attached.
last_date=panel.date.max(); future_dates=pd.date_range(last_date+pd.Timedelta(days=1),periods=HORIZON,freq='D'); latest_history=panel.copy()
if selected in models:
 final_model=fit_candidate(selected,last_date)
 future_rows,_=recursive_forecast(final_model,latest_history,future_dates,ids)
 with open(OUT/'selected_model.pkl','wb') as f: pickle.dump(final_model,f)
 importances=getattr(final_model,'feature_importances_',None)
 if importances is not None: pd.DataFrame({'feature':FEATURES,'importance':importances}).sort_values('importance',ascending=False).to_csv(OUT/'feature_importance.csv',index=False)
else:
 final_model=None
 future_rows=seasonal_predictions(latest_history,future_dates,ids).rename(columns={'prediction':'forecast'})
if 'onpromotion' not in future_rows: future_rows=future_rows.merge(covariates,on=['date','store_nbr','family'],how='left')
future_rows['is_holiday']=future_rows.date.dt.normalize().isin(holiday_dates).astype(int)
future_rows[['date','store_nbr','family','forecast','onpromotion','is_holiday']].to_csv(OUT/'future_forecast_by_store_family.csv',index=False)
display(future_rows.head(10))
daily=future_rows.groupby('date').forecast.sum(); fig,ax=plt.subplots(figsize=(13,4)); panel.loc[panel.date>=last_date-pd.Timedelta(days=120)].groupby('date').sales.sum().plot(ax=ax,label='Historical total'); daily.plot(ax=ax,label='16-day forecast total'); ax.axvline(last_date,color='black',ls='--',label='Forecast origin'); ax.set_ylabel('Sales'); ax.legend(); plt.tight_layout(); plt.savefig(OUT/'future_forecast_total.png'); plt.show()
fig,ax=plt.subplots(figsize=(12,5)); future_rows.groupby('family').forecast.sum().sort_values(ascending=False).head(15).sort_values().plot(kind='barh',ax=ax); ax.set(title='Top forecast product families over horizon',xlabel='Forecast sales'); plt.tight_layout(); plt.savefig(OUT/'future_top_families.png'); plt.show()
meta={'python':platform.python_version(),'pandas':pd.__version__,'input':str(TRAIN_PATH),'horizon_days':HORIZON,'series_keys':['store_nbr','family'],'selected_model':selected,'validation_folds':3,'heldout_test_folds':2,'split_rule':'rolling-origin; each window 16 days; final two windows held out','promotion_source':'train.csv in backtests; test.csv for future when available; unknown future defaults to zero','missing_targets':'preserved as missing; not treated as zero','holiday_source':str(HOLIDAY_PATH),'seed':SEED}
(OUT/'run_metadata.json').write_text(json.dumps(meta,indent=2))
zip_path=Path('/kaggle/working/demand_forecasting_bundle.zip') if Path('/kaggle/working').exists() else Path('./demand_forecasting_bundle.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for f in OUT.rglob('*'):
  if f.is_file(): z.write(f,arcname='demand_forecast_output/'+str(f.relative_to(OUT)))
print('Bundle:',zip_path,'bytes:',zip_path.stat().st_size); display(Markdown('**Click to download the training bundle:**')); display(FileLink(str(zip_path)))
print('Review validation and held-out test scores, per-family errors, and known-promotion coverage before using the model.')

## Interpretation and checks before handoff

Compare each model against SeasonalNaive_7 on the exact same five windows; store_family_metrics_by_fold.csv reports family-level and store-level results. Choose by the first three validation windows only; the last two test windows are a final unbiased check. Inspect scores per product family and store before making a deployment decision. The four missing calendar dates remain missing targets, not zero sales. For future promotion values, check coverage in test.csv; the fallback is zero only when no known plan is available. The holiday feature currently flags any date listed in holidays_events.csv across locations; location-specific holiday handling can be added if it materially improves errors.